# A1 · Acquire — put a little real data on Drive, and prove it is the right data

**One job.** Fetch a small, explicit amount of data for one or more tracks into a persistent Drive
store, verify it against what each track expects, and write a manifest the later scripts read.

It does not train, does not preprocess, and does not touch NeuralBench.

---

**Why EEGDash and not NeuralBench.** `neuralbench --download` fetches whole corpora, with no subset
option. Track 4's is 770 GB. EEGDash queries individual records, so a few hundred megabytes is
reachable. Everything here goes through EEGDash.

**Why Drive.** Colab wipes `/content` between sessions. Anything cached there is re-fetched every
run, which is what made the old smoke test slow. The store persists, so a second run of this
notebook costs nothing unless you ask for more.

**Re-running is safe.** It inventories first and asks before fetching. Reuse is the default.

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'   # the folder you created

TRACKS_TO_DO = 'all'    # 'all', or e.g. '3' or '2,3'
BUDGET_MB = 40          # per track. Start small; top up later.

print(f'root   {ROOT}')
print(f'tracks {TRACKS_TO_DO}')
print(f'budget {BUDGET_MB} MB per track')

In [ ]:
import sys, subprocess
from pathlib import Path

SRC = Path('/content/repo/src')
if Path('../src/tracks.py').exists():
    SRC = Path('../src').resolve()
elif not SRC.exists():
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/AGRamirezz/Neurips26-eeg-foundation-model.git',
                    '/content/repo'], check=True)
sys.path.insert(0, str(SRC))
print('helpers:', sorted(f.name for f in SRC.glob('*.py')))

In [ ]:
%pip install -q 'eegdash>=0.9.1' mne

import importlib
for mod in ('eegdash', 'mne'):
    importlib.import_module(mod)
    print(f'{mod} ok')

In [ ]:
import os

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

import tracks, acquire, datastore, miniload

ROOT = Path(ROOT)
ROOT.mkdir(parents=True, exist_ok=True)
shared = tracks.shared_paths(ROOT)
os.environ['HF_HOME'] = str(shared['hf'])

selected = (list(tracks.TRACKS) if TRACKS_TO_DO.strip() == 'all'
            else [k.strip() for k in TRACKS_TO_DO.split(',')])
for k in selected:
    tracks.get(k)          # fails now rather than mid-download
print('will process:', [tracks.get(k).slug for k in selected])

## What is already here

Before fetching anything. An empty row means that track has never been seeded.

In [ ]:
print(acquire.overview(ROOT, tracks.TRACKS))

## Fetch

For each selected track: query the catalogue, verify the record metadata, pick a subset under the
budget spread across subjects, then ask before downloading.

`EEGDashDataset` is lazy and only fetches when a recording is touched, so the download is explicit
rather than a side effect of constructing it.

In [ ]:
from eegdash import EEGDash, EEGDashDataset
import mne

mne.set_log_level('ERROR')
results = {}

for key in selected:
    t = tracks.get(key)
    p = tracks.paths(ROOT, t)
    os.environ['EEGDASH_CACHE_DIR'] = str(p['raw'])
    print('=' * 72)
    print(f'Track {t.key} - {t.name}   [{t.eegdash_id}]  -> {p["raw"]}')
    print('=' * 72)

    records = [r for r in EEGDash().find({'dataset': t.eegdash_id})
               if r['bids_relpath'].startswith('sub-')]
    print(acquire.format_checks(acquire.verify_records(records, t), 'catalogue:'))

    sel = miniload.select_under_budget(
        records, budget_mb=BUDGET_MB, spread_by='subject',
        exclude_sessions=t.exclude_sessions)
    print(f'\nselection: {len(sel)} records, ~{sel.est_mb:.0f} MB, '
          f'{len({r["subject"] for r in sel.records})} subjects')

    inv = datastore.inventory(p['raw'])
    action, delta = datastore.choose(inv, t.eegdash_id, BUDGET_MB)
    results[key] = dict(track=t, paths=p, sel=sel, action=action)
    print()

In [ ]:
# The fetch itself. Touching .raw is what pulls a recording.
import time

for key, r in results.items():
    t, p, sel, action = r['track'], r['paths'], r['sel'], r['action']
    if action == 'reuse':
        print(f'{t.slug}: reusing what is on disk')
        continue

    os.environ['EEGDASH_CACHE_DIR'] = str(p['raw'])
    print(f'{t.slug}: fetching {len(sel)} records...')
    t0 = time.time()
    ds = EEGDashDataset(records=sel.records, cache_dir=p['raw'])
    fetched = []
    for i, d in enumerate(ds.datasets, 1):
        try:
            _ = d.raw
            fetched.append(d)
        except Exception as e:
            print(f'  record {i} failed: {type(e).__name__}: {str(e)[:90]}')
    r['fetched'] = fetched
    print(f'  {len(fetched)}/{len(sel)} in {time.time() - t0:.0f}s')

## Verify and record

Metadata checks pass on things that were never downloaded, so this re-checks against an actual
signal: sampling rate, channel count, whether a montage is present, whether any events exist.

A failed check is written into the manifest rather than raising. The point is to know the state of
the data, not to stop at the first surprise.

In [ ]:
for key, r in results.items():
    t, p = r['track'], r['paths']
    fetched = r.get('fetched') or []
    inv = datastore.inventory(p['raw'])
    on_disk_mb = sum(e.mb for e in inv)

    checks = acquire.verify_records(r['sel'].records, t)
    if fetched:
        checks += acquire.verify_recording(fetched[0].raw, t)
    else:
        existing = sorted(p['raw'].rglob('*.edf')) + sorted(p['raw'].rglob('*.bdf'))
        if existing:
            checks += acquire.verify_recording(
                mne.io.read_raw(existing[0], preload=False, verbose='ERROR'), t)

    print(acquire.format_checks(checks, f'{t.slug}:'))
    acquire.write_manifest(p['manifest'], acquire.build_manifest(
        t, r['sel'].records, checks, mb=on_disk_mb,
        notes=f'budget {BUDGET_MB} MB, action {r["action"]}'))
    print(f'  manifest -> {p["manifest"]}\n')

## Where things stand

In [ ]:
print(acquire.overview(ROOT, tracks.TRACKS))
print()
for key in selected:
    t = tracks.get(key)
    print(f'{t.slug:14} target: {t.target_kind}')

### Topping up later

Re-run with a larger `BUDGET_MB` and answer `a` when asked. Selection is deterministic and nested,
so a 40 MB set is a subset of an 80 MB one: the extra records are added, nothing is re-fetched.

Next: **A2 · Inspect**, which checks metadata and labels are actually usable and plots the neural
data against its paired target.